# Session 2 — Complete Model & Pretraining**Duration:** 75 min  **Repository:** [MiniMind-Colab](https://github.com/your-org/minimind-colab)| Part | Topic | Time ||------|-------|------|| A | Completing the Model — FeedForward, MOE, MiniMindBlock, CausalLM | 20 min || B | Pretraining — PretrainDataset, training loop with AMP | 50 min || — | Wrap-up & Q\&A | 5 min |

In [ ]:
# Cell 2 — Environment setup!pip install torch transformers datasets --quietimport math, json, os, timeimport torchimport torch.nn as nnimport torch.nn.functional as Ffrom torch import optimfrom torch.utils.data import Dataset, DataLoaderfrom contextlib import nullcontextfrom transformers.activations import ACT2FNfrom transformers import PreTrainedModel, GenerationMixin, PretrainedConfigfrom transformers.modeling_outputs import MoeCausalLMOutputWithPastdevice = 'cuda' if torch.cuda.is_available() else 'cpu'print(f'PyTorch {torch.__version__}  CUDA available: {torch.cuda.is_available()}')

## Part A — Completing the Model (20 min)In Session 1 we built RMSNorm, RoPE, and GQA Attention.Now we complete the transformer by adding:1. **FeedForward (SwiGLU)** — SiLU-gated feed-forward network2. **MOEFeedForward** — Mixture-of-Experts variant (provided for reference)3. **MiniMindBlock** — Pre-norm transformer block with residual connections4. **MiniMindForCausalLM** — Full causal language model with weight tying

### Prerequisites from Session 1We re-define RMSNorm, RoPE, Attention, and the config from Session 1.These are provided — no changes needed.

In [ ]:
# Cell — Prerequisites: RMSNorm, RoPE, Attention (from Session 1)class RMSNorm(torch.nn.Module):    def __init__(self, dim, eps=1e-5):        super().__init__()        self.eps = eps        self.weight = nn.Parameter(torch.ones(dim))    def norm(self, x):        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)    def forward(self, x):        return (self.weight * self.norm(x.float())).type_as(x)def precompute_freqs_cis(dim, end=32*1024, rope_base=1e6):    freqs = 1.0 / (rope_base ** (torch.arange(0, dim, 2)[:(dim // 2)].float() / dim))    t = torch.arange(end)    freqs = torch.outer(t, freqs).float()    freqs_cos = torch.cat([torch.cos(freqs), torch.cos(freqs)], dim=-1)    freqs_sin = torch.cat([torch.sin(freqs), torch.sin(freqs)], dim=-1)    return freqs_cos, freqs_sindef apply_rotary_pos_emb(q, k, cos, sin, unsqueeze_dim=1):    def rotate_half(x):        return torch.cat((-x[..., x.shape[-1] // 2:], x[..., :x.shape[-1] // 2]), dim=-1)    q_embed = ((q * cos.unsqueeze(unsqueeze_dim)) + (rotate_half(q) * sin.unsqueeze(unsqueeze_dim))).to(q.dtype)    k_embed = ((k * cos.unsqueeze(unsqueeze_dim)) + (rotate_half(k) * sin.unsqueeze(unsqueeze_dim))).to(k.dtype)    return q_embed, k_embeddef repeat_kv(x, n_rep):    bs, slen, num_kv_heads, head_dim = x.shape    if n_rep == 1:        return x    return (x[:, :, :, None, :]            .expand(bs, slen, num_kv_heads, n_rep, head_dim)            .reshape(bs, slen, num_kv_heads * n_rep, head_dim))class MiniMindConfig(PretrainedConfig):    model_type = 'minimind'    def __init__(self, hidden_size=768, num_hidden_layers=8, use_moe=False, **kwargs):        super().__init__(**kwargs)        self.hidden_size = hidden_size        self.num_hidden_layers = num_hidden_layers        self.use_moe = use_moe        self.dropout = kwargs.get('dropout', 0.0)        self.vocab_size = kwargs.get('vocab_size', 6400)        self.bos_token_id = kwargs.get('bos_token_id', 1)        self.eos_token_id = kwargs.get('eos_token_id', 2)        self.flash_attn = kwargs.get('flash_attn', True)        self.num_attention_heads = kwargs.get('num_attention_heads', 8)        self.num_key_value_heads = kwargs.get('num_key_value_heads', 4)        self.head_dim = kwargs.get('head_dim', self.hidden_size // self.num_attention_heads)        self.hidden_act = kwargs.get('hidden_act', 'silu')        self.intermediate_size = kwargs.get('intermediate_size', math.ceil(hidden_size * math.pi / 64) * 64)        self.max_position_embeddings = kwargs.get('max_position_embeddings', 32768)        self.rms_norm_eps = kwargs.get('rms_norm_eps', 1e-6)        self.rope_theta = kwargs.get('rope_theta', 1e6)        self.num_experts = kwargs.get('num_experts', 4)        self.num_experts_per_tok = kwargs.get('num_experts_per_tok', 1)        self.moe_intermediate_size = kwargs.get('moe_intermediate_size', self.intermediate_size)        self.norm_topk_prob = kwargs.get('norm_topk_prob', True)        self.router_aux_loss_coef = kwargs.get('router_aux_loss_coef', 5e-4)class Attention(nn.Module):    def __init__(self, config):        super().__init__()        self.num_key_value_heads = config.num_attention_heads if config.num_key_value_heads is None else config.num_key_value_heads        self.n_local_heads = config.num_attention_heads        self.n_local_kv_heads = self.num_key_value_heads        self.n_rep = self.n_local_heads // self.n_local_kv_heads        self.head_dim = config.head_dim        self.q_proj = nn.Linear(config.hidden_size, config.num_attention_heads * self.head_dim, bias=False)        self.k_proj = nn.Linear(config.hidden_size, self.num_key_value_heads * self.head_dim, bias=False)        self.v_proj = nn.Linear(config.hidden_size, self.num_key_value_heads * self.head_dim, bias=False)        self.o_proj = nn.Linear(config.num_attention_heads * self.head_dim, config.hidden_size, bias=False)        self.q_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)        self.k_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)        self.attn_dropout = nn.Dropout(config.dropout)        self.resid_dropout = nn.Dropout(config.dropout)        self.dropout = config.dropout        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention') and config.flash_attn    def forward(self, x, position_embeddings, past_key_value=None, use_cache=False, attention_mask=None):        bsz, seq_len, _ = x.shape        xq, xk, xv = self.q_proj(x), self.k_proj(x), self.v_proj(x)        xq = xq.view(bsz, seq_len, self.n_local_heads, self.head_dim)        xk = xk.view(bsz, seq_len, self.n_local_kv_heads, self.head_dim)        xv = xv.view(bsz, seq_len, self.n_local_kv_heads, self.head_dim)        xq, xk = self.q_norm(xq), self.k_norm(xk)        cos, sin = position_embeddings        xq, xk = apply_rotary_pos_emb(xq, xk, cos, sin)        if past_key_value is not None:            xk = torch.cat([past_key_value[0], xk], dim=1)            xv = torch.cat([past_key_value[1], xv], dim=1)        past_kv = (xk, xv) if use_cache else None        xq = xq.transpose(1, 2)        xk = repeat_kv(xk, self.n_rep).transpose(1, 2)        xv = repeat_kv(xv, self.n_rep).transpose(1, 2)        if self.flash and (seq_len > 1) and (past_key_value is None) and (attention_mask is None or torch.all(attention_mask == 1)):            output = F.scaled_dot_product_attention(xq, xk, xv, dropout_p=self.dropout if self.training else 0.0, is_causal=True)        else:            scores = (xq @ xk.transpose(-2, -1)) / math.sqrt(self.head_dim)            scores[:, :, :, -seq_len:] += torch.full((seq_len, seq_len), float('-inf'), device=scores.device).triu(1)            if attention_mask is not None:                scores += (1.0 - attention_mask.unsqueeze(1).unsqueeze(2)) * -1e9            output = self.attn_dropout(F.softmax(scores.float(), dim=-1).type_as(xq)) @ xv        output = output.transpose(1, 2).reshape(bsz, seq_len, -1)        output = self.resid_dropout(self.o_proj(output))        return output, past_kvprint('Session 1 prerequisites loaded ✓')

### 1. FeedForward — SiLU-Gated (SwiGLU) NetworkThe FeedForward network uses a **gated** architecture (SwiGLU):$$\text{FFN}(x) = W_{\text{down}} \cdot (\text{SiLU}(W_{\text{gate}} x) \odot W_{\text{up}} x)$$Three weight matrices:- `gate_proj`: projects to intermediate size, passed through SiLU activation- `up_proj`: projects to intermediate size (no activation)- `down_proj`: projects the element-wise product back to hidden size**Why gating?** — The element-wise multiplication of two different projectionsgives the network more expressiveness per parameter compared to a simple MLP.

In [ ]:
# Cell — FeedForward (SwiGLU) implementationclass FeedForward(nn.Module):    def __init__(self, config, intermediate_size=None):        super().__init__()        intermediate_size = intermediate_size or config.intermediate_size        self.gate_proj = nn.Linear(config.hidden_size, intermediate_size, bias=False)        self.down_proj = nn.Linear(intermediate_size, config.hidden_size, bias=False)        self.up_proj = nn.Linear(config.hidden_size, intermediate_size, bias=False)        self.act_fn = ACT2FN[config.hidden_act]    def forward(self, x):        # ============================================================        # TODO: Implement the SiLU-gated (SwiGLU) forward pass        #        # Compute the gated feed-forward transformation.        #        # Input:  x — shape (batch, seq_len, hidden_size)        # Output: tensor of same shape (batch, seq_len, hidden_size)        #        # Mathematical intuition:        #   FFN(x) = down_proj( act_fn(gate_proj(x)) * up_proj(x) )        #   The gate branch applies SiLU activation, then is multiplied        #   element-wise with the up branch before projecting back down.        #        # Hint: self.act_fn, self.gate_proj, self.up_proj, self.down_proj        # ============================================================        raise NotImplementedError("Fill in the TODO above")print('FeedForward class defined ✓')

In [ ]:
# Cell — ✅ Milestone 1: FFN forward shape checkconfig = MiniMindConfig()ffn = FeedForward(config)x = torch.randn(2, 32, config.hidden_size)with torch.no_grad():    out = ffn(x)expected_shape = (2, 32, config.hidden_size)assert out.shape == expected_shape, (    f'FFN output shape mismatch: {out.shape} vs {expected_shape}')print(f'✅ Milestone 1 passed — FFN output shape: {out.shape}')print(f'   Input:  (2, 32, {config.hidden_size})')print(f'   Output: {out.shape}')print(f'   Intermediate size: {config.intermediate_size}')

### 2. MOEFeedForward — Mixture of Experts (Reference)When `config.use_moe=True`, each transformer block uses a Mixture-of-ExpertsFFN instead of a single FFN. A learned **router** selects which expert(s)process each token.This is provided for reference — we will use `use_moe=False` in our training.

In [ ]:
# Cell — MOEFeedForward (provided — no changes needed)class MOEFeedForward(nn.Module):    def __init__(self, config):        super().__init__()        self.config = config        self.gate = nn.Linear(config.hidden_size, config.num_experts, bias=False)        self.experts = nn.ModuleList([            FeedForward(config, intermediate_size=config.moe_intermediate_size)            for _ in range(config.num_experts)        ])        self.act_fn = ACT2FN[config.hidden_act]    def forward(self, x):        batch_size, seq_len, hidden_dim = x.shape        x_flat = x.view(-1, hidden_dim)        scores = F.softmax(self.gate(x_flat), dim=-1)        topk_weight, topk_idx = torch.topk(scores, k=self.config.num_experts_per_tok, dim=-1, sorted=False)        if self.config.norm_topk_prob:            topk_weight = topk_weight / (topk_weight.sum(dim=-1, keepdim=True) + 1e-20)        y = torch.zeros_like(x_flat)        for i, expert in enumerate(self.experts):            mask = (topk_idx == i)            if mask.any():                token_idx = mask.any(dim=-1).nonzero().flatten()                weight = topk_weight[mask].view(-1, 1)                y.index_add_(0, token_idx, (expert(x_flat[token_idx]) * weight).to(y.dtype))            elif self.training:                y[0, 0] += 0 * sum(p.sum() for p in expert.parameters())        if self.training and self.config.router_aux_loss_coef > 0:            load = F.one_hot(topk_idx, self.config.num_experts).float().mean(0)            self.aux_loss = (load * scores.mean(0)).sum() * self.config.num_experts * self.config.router_aux_loss_coef        else:            self.aux_loss = scores.new_zeros(1).squeeze()        return y.view(batch_size, seq_len, hidden_dim)print('MOEFeedForward class defined ✓')

### 3. MiniMindBlock — Pre-Norm Transformer BlockEach transformer block follows the **pre-norm** residual pattern:```h = h + Attention(LayerNorm(h))h = h + FFN(LayerNorm(h))```This is more stable than post-norm (LayerNorm after residual add).Note how the residual is saved *before* normalization and added *after*the sub-layer output.

In [ ]:
# Cell — MiniMindBlock implementationclass MiniMindBlock(nn.Module):    def __init__(self, layer_id, config):        super().__init__()        self.self_attn = Attention(config)        self.input_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)        self.post_attention_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)        self.mlp = FeedForward(config) if not config.use_moe else MOEFeedForward(config)    def forward(self, hidden_states, position_embeddings, past_key_value=None, use_cache=False, attention_mask=None):        # ============================================================        # TODO: Implement the pre-norm transformer block forward pass        #        # Apply the pre-norm residual pattern:        #   1. Save residual, apply input_layernorm, then self-attention        #   2. Add residual to attention output (first residual connection)        #   3. Apply post_attention_layernorm, then MLP/FFN        #   4. Add to hidden_states (second residual connection)        #        # Input:  hidden_states — (batch, seq_len, hidden_size)        # Output: (hidden_states, present_key_value) — same shape + KV cache        #        # Mathematical intuition:        #   h = h + Attention(LayerNorm(h))        #   h = h + FFN(LayerNorm(h))        #        # Hint: self.input_layernorm, self.self_attn, self.post_attention_layernorm, self.mlp        #   self.self_attn returns (output, present_key_value)        # ============================================================        raise NotImplementedError("Fill in the TODO above")print('MiniMindBlock class defined ✓')

### 4. MiniMindModel & MiniMindForCausalLM**MiniMindModel** stacks N transformer blocks with:- Token embeddings + dropout- Precomputed RoPE frequencies as buffers- Final RMSNorm**MiniMindForCausalLM** wraps the model with an `lm_head` (linear projection to vocab)and **weight tying** (embedding weights = lm_head weights).The forward pass computes cross-entropy loss with **label shifting**:predict token `t+1` from position `t`.

In [ ]:
# Cell — MiniMindModel (provided — no changes needed)class MiniMindModel(nn.Module):    def __init__(self, config):        super().__init__()        self.config = config        self.vocab_size = config.vocab_size        self.num_hidden_layers = config.num_hidden_layers        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)        self.dropout = nn.Dropout(config.dropout)        self.layers = nn.ModuleList([MiniMindBlock(l, config) for l in range(self.num_hidden_layers)])        self.norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)        freqs_cos, freqs_sin = precompute_freqs_cis(            dim=config.head_dim, end=config.max_position_embeddings, rope_base=config.rope_theta        )        self.register_buffer('freqs_cos', freqs_cos, persistent=False)        self.register_buffer('freqs_sin', freqs_sin, persistent=False)    def forward(self, input_ids, attention_mask=None, past_key_values=None, use_cache=False, **kwargs):        batch_size, seq_length = input_ids.shape        if hasattr(past_key_values, 'layers'):            past_key_values = None        past_key_values = past_key_values or [None] * len(self.layers)        start_pos = past_key_values[0][0].shape[1] if past_key_values[0] is not None else 0        hidden_states = self.dropout(self.embed_tokens(input_ids))        position_embeddings = (            self.freqs_cos[start_pos:start_pos + seq_length],            self.freqs_sin[start_pos:start_pos + seq_length],        )        presents = []        for layer, past_kv in zip(self.layers, past_key_values):            hidden_states, present = layer(                hidden_states, position_embeddings,                past_key_value=past_kv, use_cache=use_cache, attention_mask=attention_mask            )            presents.append(present)        hidden_states = self.norm(hidden_states)        aux_loss = sum(            [l.mlp.aux_loss for l in self.layers if isinstance(l.mlp, MOEFeedForward)],            hidden_states.new_zeros(1).squeeze()        )        return hidden_states, presents, aux_lossprint('MiniMindModel class defined ✓')

In [ ]:
# Cell — MiniMindForCausalLM implementationclass MiniMindForCausalLM(PreTrainedModel, GenerationMixin):    config_class = MiniMindConfig    def __init__(self, config=None):        self.config = config or MiniMindConfig()        super().__init__(self.config)        self.model = MiniMindModel(self.config)        self.lm_head = nn.Linear(self.config.hidden_size, self.config.vocab_size, bias=False)        self.model.embed_tokens.weight = self.lm_head.weight  # weight tying    def forward(self, input_ids, labels=None, **kwargs):        hidden_states, past_key_values, aux_loss = self.model(input_ids, **kwargs)        logits = self.lm_head(hidden_states)        loss = None        if labels is not None:            # ============================================================            # TODO: Compute cross-entropy loss with label shifting            #            # The model predicts the NEXT token at each position, so we            # need to shift logits and labels before computing the loss.            #            # Input:  logits — (batch, seq_len, vocab_size)            #         labels — (batch, seq_len)            # Output: loss — scalar cross-entropy loss            #            # Mathematical intuition:            #   x = logits[..., :-1, :]   # predictions for tokens 0..T-2            #   y = labels[..., 1:]        # targets are tokens 1..T-1            #   loss = CrossEntropy(x.view(-1, V), y.view(-1), ignore_index=-100)            #            # Hint: .contiguous(), .view(-1, vocab_size), F.cross_entropy            #   Use ignore_index=-100 to skip padding tokens            # ============================================================            raise NotImplementedError("Fill in the TODO above")        return MoeCausalLMOutputWithPast(loss=loss, aux_loss=aux_loss, logits=logits,                                         past_key_values=past_key_values, hidden_states=hidden_states)print('MiniMindForCausalLM class defined ✓')

In [ ]:
# Cell — Instantiate model and test forward passconfig = MiniMindConfig(hidden_size=768, num_hidden_layers=8)model = MiniMindForCausalLM(config).to(device)total_params = sum(p.numel() for p in model.parameters()) / 1e6print(f'Model parameters: {total_params:.2f}M')# Quick forward pass testdummy_ids = torch.randint(0, config.vocab_size, (2, 32)).to(device)dummy_labels = dummy_ids.clone()with torch.no_grad():    out = model(dummy_ids, labels=dummy_labels)print(f'Logits shape: {out.logits.shape}')print(f'Loss: {out.loss.item():.4f}')assert out.logits.shape == (2, 32, config.vocab_size), f'Logits shape mismatch: {out.logits.shape}'assert out.loss is not None, 'Loss should not be None when labels are provided'print('✓ Model forward pass works correctly')

## Part B — Pretraining (50 min)Now we'll set up the data pipeline and training loop to pretrain our model.Key components:1. **PretrainDataset** — tokenize text, add BOS/EOS, pad, create labels2. **Learning rate schedule** — cosine annealing with warmup3. **Training loop** — mixed-precision (AMP), gradient accumulation, grad clipping

### 5. PretrainDatasetFor pretraining, each sample is a text document that gets:1. Tokenized (truncated to `max_length - 2` to leave room for special tokens)2. Wrapped with `BOS` (beginning of sequence) and `EOS` (end of sequence)3. Padded to `max_length` with `pad_token_id`4. Labels are a copy of input_ids, but with pad positions set to `-100`   (so the loss ignores padding tokens)

In [ ]:
# Cell — PretrainDataset implementationclass PretrainDataset(Dataset):    def __init__(self, data_path, tokenizer, max_length=512):        super().__init__()        self.tokenizer = tokenizer        self.max_length = max_length        # For this notebook, we create synthetic data inline        self.samples = [            {'text': f'MiniMind is a tiny language model for teaching. Sample {i}.'}            for i in range(200)        ]    def __len__(self):        return len(self.samples)    def __getitem__(self, index):        sample = self.samples[index]        # ============================================================        # TODO: Tokenize the text and prepare input_ids + labels        #        # Steps:        #   1. Tokenize sample['text'] with self.tokenizer (no special tokens,        #      truncate to max_length-2)        #   2. Prepend BOS token and append EOS token        #   3. Pad with pad_token_id to self.max_length        #   4. Convert to torch.long tensor        #   5. Create labels: clone input_ids, set pad positions to -100        #        # Input:  sample — dict with key 'text'        # Output: (input_ids, labels) — both shape (max_length,)        #        # Mathematical intuition:        #   input_ids = [BOS] + tokens + [EOS] + [PAD]*remaining        #   labels    = same, but PAD positions → -100        #        # Hint: self.tokenizer(...).input_ids, self.tokenizer.bos_token_id,        #   self.tokenizer.eos_token_id, self.tokenizer.pad_token_id        # ============================================================        raise NotImplementedError("Fill in the TODO above")print('PretrainDataset class defined ✓')

In [ ]:
# Cell — Tokenizer & DataLoader setupfrom transformers import AutoTokenizertokenizer = AutoTokenizer.from_pretrained('HuggingFaceTB/cosmo2-tokenizer', trust_remote_code=True)# Ensure special tokens are setif tokenizer.pad_token is None:    tokenizer.pad_token = tokenizer.eos_tokenMAX_SEQ_LEN = 128BATCH_SIZE = 8train_ds = PretrainDataset(data_path=None, tokenizer=tokenizer, max_length=MAX_SEQ_LEN)loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)print(f'Dataset size: {len(train_ds)}')print(f'Batches: {len(loader)}')print(f'Vocab size: {tokenizer.vocab_size}')

In [ ]:
# Cell — ✅ Milestone 2: DataLoader batch verificationbatch_ids, batch_labels = next(iter(loader))assert batch_ids.shape == (BATCH_SIZE, MAX_SEQ_LEN), (    f'input_ids shape mismatch: {batch_ids.shape} vs ({BATCH_SIZE}, {MAX_SEQ_LEN})')assert batch_labels.shape == (BATCH_SIZE, MAX_SEQ_LEN), (    f'labels shape mismatch: {batch_labels.shape} vs ({BATCH_SIZE}, {MAX_SEQ_LEN})')# Verify pad tokens have labels == -100pad_mask = (batch_ids == tokenizer.pad_token_id)if pad_mask.any():    assert (batch_labels[pad_mask] == -100).all(), 'Pad tokens should have labels == -100'    print(f'  Pad token positions correctly masked to -100')print(f'✅ Milestone 2 passed — DataLoader batch shapes correct')print(f'   input_ids:  {batch_ids.shape}')print(f'   labels:     {batch_labels.shape}')print(f'   Pad tokens: {pad_mask.sum().item()} out of {pad_mask.numel()}')

### 6. Learning Rate Schedule & Training LoopWe use **cosine annealing** with a floor factor:$$\text{lr}(t) = \text{lr}_{\max} \cdot \left(0.1 + 0.45 \cdot \left(1 + \cos\left(\frac{\pi \cdot t}{T}\right)\right)\right)$$This starts near `lr_max`, decays smoothly to `0.1 * lr_max`.

In [ ]:
# Cell — Learning rate schedule (provided)def get_lr(current_step, total_steps, lr):    return lr * (0.1 + 0.45 * (1 + math.cos(math.pi * current_step / total_steps)))# Visualize the scheduletotal_steps = 100lrs = [get_lr(s, total_steps, 5e-4) for s in range(total_steps)]print(f'LR range: {min(lrs):.6f} — {max(lrs):.6f}')print(f'Start LR: {lrs[0]:.6f}')print(f'End LR:   {lrs[-1]:.6f}')print('get_lr function defined ✓')

### Training Loop with Mixed Precision & Gradient AccumulationKey techniques:- **AMP (Automatic Mixed Precision):** reduces memory by using float16/bfloat16- **GradScaler:** prevents underflow in float16 gradients- **Gradient accumulation:** simulate larger batch sizes without more memory- **Gradient clipping:** prevent exploding gradients (max norm = 1.0)

In [ ]:
# Cell — Training setup# Use a small model for notebook trainingconfig = MiniMindConfig(    hidden_size=128,    num_hidden_layers=2,    num_attention_heads=4,    num_key_value_heads=2,    vocab_size=tokenizer.vocab_size,)torch.manual_seed(42)model = MiniMindForCausalLM(config).to(device)total_params = sum(p.numel() for p in model.parameters()) / 1e6print(f'Training model: {total_params:.2f}M parameters')# Hyperparameterslearning_rate = 5e-4accumulation_steps = 2num_epochs = 1MAX_SEQ_LEN = 128BATCH_SIZE = 8# Recreate dataset and loader with the correct vocabtrain_ds = PretrainDataset(data_path=None, tokenizer=tokenizer, max_length=MAX_SEQ_LEN)loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)# Optimizeroptimizer = optim.AdamW(model.parameters(), lr=learning_rate)# Mixed precision setupdtype_str = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16'use_amp = torch.cuda.is_available()dtype = torch.bfloat16 if dtype_str == 'bfloat16' else torch.float16autocast_ctx = nullcontext() if not use_amp else torch.cuda.amp.autocast(dtype=dtype)scaler = torch.cuda.amp.GradScaler(enabled=(dtype_str == 'float16' and use_amp))total_steps = len(loader) * num_epochsprint(f'Total training steps: {total_steps}')print(f'Mixed precision: {dtype_str if use_amp else "disabled (CPU)"}')

In [ ]:
# Cell — Training loopmodel.train()losses = []log_interval = 10global_step = 0for epoch in range(num_epochs):    for step, (input_ids, labels) in enumerate(loader):        input_ids = input_ids.to(device)        labels = labels.to(device)        global_step += 1        # ============================================================        # TODO: Implement the training loop inner body        #        # Steps to implement:        #   1. Compute cosine LR with get_lr() and update optimizer        #   2. Forward pass inside autocast_ctx context manager        #   3. Scale loss by accumulation_steps        #   4. Backward pass with scaler.scale(loss).backward()        #   5. Every accumulation_steps: unscale, clip grads, step, zero_grad        #        # Input:  input_ids, labels — (batch, seq_len) on device        # Output: current_loss — scalar float for logging        #        # Mathematical intuition:        #   lr(t) = lr_max * (0.1 + 0.45 * (1 + cos(π * t / T)))        #   loss = (model_loss + aux_loss) / accumulation_steps        #   Gradient accumulation simulates larger effective batch size        #        # Hint: get_lr, optimizer.param_groups, autocast_ctx, model(),        #   scaler.scale, scaler.unscale_, clip_grad_norm_, scaler.step        # ============================================================        raise NotImplementedError("Fill in the TODO above")        # Logging (keep this — uses `current_loss` from your code above)        losses.append(current_loss)        if global_step % log_interval == 0 or global_step == 1:            print(f'Step {global_step:>4d}/{total_steps} | Loss: {current_loss:.4f} | LR: {lr:.6f}')# Final cleanup for any remaining gradientsif global_step % accumulation_steps != 0:    scaler.unscale_(optimizer)    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)    scaler.step(optimizer)    scaler.update()    optimizer.zero_grad(set_to_none=True)print(f'\nTraining complete. Final loss: {losses[-1]:.4f}')

In [ ]:
# Cell — ✅ Milestone 3: Loss starts high and decreasesfirst_losses = losses[:5]last_losses = losses[-5:]first_avg = sum(first_losses) / len(first_losses)last_avg = sum(last_losses) / len(last_losses)print(f'First 5 losses avg: {first_avg:.4f}')print(f'Last 5 losses avg:  {last_avg:.4f}')print(f'Loss decrease:      {first_avg - last_avg:.4f}')# Initial loss for random weights should be ~ln(vocab_size) ≈ 8.7 for vocab=6400# For larger vocabs the initial loss will be higherexpected_initial = math.log(config.vocab_size)print(f'\nExpected initial loss ≈ ln({config.vocab_size}) = {expected_initial:.2f}')assert last_avg < first_avg, (    f'Loss did not decrease! First avg: {first_avg:.4f}, Last avg: {last_avg:.4f}')assert len(losses) >= 20, f'Not enough training steps: {len(losses)}'print(f'✅ Milestone 3 passed — Loss decreased from {first_avg:.4f} to {last_avg:.4f}')

## Wrap-up (5 min)### What we built today| Component | Key idea ||-----------|----------|| **FeedForward (SwiGLU)** | Gated FFN: `down(act(gate(x)) * up(x))` || **MiniMindBlock** | Pre-norm residual: `h + SubLayer(Norm(h))` || **MiniMindForCausalLM** | Weight tying + shifted cross-entropy loss || **PretrainDataset** | BOS/EOS wrapping, padding, label masking || **Training loop** | AMP, GradScaler, cosine LR, gradient accumulation |### Next session- Supervised Fine-Tuning (SFT) with chat templates- LoRA: parameter-efficient fine-tuning### References- [SwiGLU paper (Shazeer 2020)](https://arxiv.org/abs/2002.05202)- [Mixed precision training](https://arxiv.org/abs/1710.03740)- [Weight tying](https://arxiv.org/abs/1608.05859)